In [1]:
# pip install gensim

In [1]:
import re, time, math, ssl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [4]:
import ssl
import gensim
from sklearn.decomposition import PCA
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("gensim", gensim.__version__, "| tensorflow", tf.__version__)

gensim 4.4.0 | tensorflow 2.21.0


In [5]:
import gensim.downloader as api
 
try:
    ssl._create_default_https_context = ssl._create_unverified_context   # SSL workaround
except AttributeError:
    pass
start = time.time()
glove = api.load("glove-wiki-gigaword-50")       # first time: downloads; later: loads from disk
print("Words in GloVe :", len(glove.key_to_index))
print("Numbers per word:", glove.vector_size)
print("Loaded in", round(time.time() - start, 1), "seconds")

Words in GloVe : 400000
Numbers per word: 50
Loaded in 46.3 seconds


In [6]:
glove["king"]

array([ 0.50451 ,  0.68607 , -0.59517 , -0.022801,  0.60046 , -0.13498 ,
       -0.08813 ,  0.47377 , -0.61798 , -0.31012 , -0.076666,  1.493   ,
       -0.034189, -0.98173 ,  0.68229 ,  0.81722 , -0.51874 , -0.31503 ,
       -0.55809 ,  0.66421 ,  0.1961  , -0.13495 , -0.11476 , -0.30344 ,
        0.41177 , -2.223   , -1.0756  , -1.0783  , -0.34354 ,  0.33505 ,
        1.9927  , -0.04234 , -0.64319 ,  0.71125 ,  0.49159 ,  0.16754 ,
        0.34344 , -0.25663 , -0.8523  ,  0.1661  ,  0.40102 ,  1.1685  ,
       -1.0137  , -0.21585 , -0.15155 ,  0.78321 , -0.91241 , -1.6106  ,
       -0.64426 , -0.51042 ], dtype=float32)

In [7]:
glove["hem"]

array([-0.15146 , -0.26358 ,  0.022398,  0.079094, -0.94758 ,  0.066723,
        0.88244 , -0.36071 , -1.0132  , -1.0602  ,  0.15531 ,  0.3545  ,
       -0.027201,  0.21427 , -0.21077 ,  0.7692  , -0.52352 , -0.17556 ,
        0.93215 , -0.22146 , -0.11738 , -0.46026 ,  0.29236 , -0.20085 ,
        0.47302 , -0.039551, -1.0325  ,  0.82278 , -0.47046 , -0.839   ,
       -0.080049,  0.43981 ,  0.19124 ,  1.3696  , -0.52844 ,  0.30983 ,
       -0.40564 ,  0.10919 ,  0.33408 ,  0.048112, -0.089205,  0.84485 ,
       -0.14552 , -0.02245 , -0.43857 , -1.5907  ,  1.2965  , -0.97772 ,
        0.28028 , -0.55971 ], dtype=float32)

In [8]:
'Queen' == 'queen' == 'QUEEN'

False

In [9]:
glove.similarity("king","king")

np.float32(1.0000001)

In [10]:
glove.similarity("king","queen")

np.float32(0.78390425)

In [11]:
glove.most_similar("dog")

[('cat', 0.9218004941940308),
 ('dogs', 0.8513158559799194),
 ('horse', 0.7907583713531494),
 ('puppy', 0.7754920721054077),
 ('pet', 0.7724708318710327),
 ('rabbit', 0.7720814347267151),
 ('pig', 0.7490062117576599),
 ('snake', 0.7399188876152039),
 ('baby', 0.7395570278167725),
 ('bite', 0.7387937307357788)]

In [12]:
glove.most_similar("hem")

[('tippy', 0.714095950126648),
 ('aksai', 0.7056282758712769),
 ('ruffles', 0.6817495226860046),
 ('straps', 0.6709622740745544),
 ('ponging', 0.6685469746589661),
 ('shimmy', 0.6650341749191284),
 ('pant', 0.660407304763794),
 ('braided', 0.6582920551300049),
 ('pleats', 0.6524637937545776),
 ('sarong', 0.6412901282310486)]

In [13]:
print("king - man + woman   :", glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3))

print("paris - france + italy:", glove.most_similar(positive=["paris", "italy"], negative=["france"], topn=3))
 
# Which word does not belong?

print("Odd one out:", glove.doesnt_match(["movie", "film", "actor", "banana"]))

king - man + woman   : [('queen', 0.8523604273796082), ('throne', 0.7664334177970886), ('prince', 0.7592144012451172)]
paris - france + italy: [('rome', 0.8465589284896851), ('milan', 0.7766007781028748), ('turin', 0.7666355967521667)]
Odd one out: banana


In [14]:
glove.most_similar(positive=["buddha", "everest", "country"])

[('earth', 0.6816374659538269),
 ('tibet', 0.669999361038208),
 ('nepal', 0.6567659378051758),
 ('himalayas', 0.6524245142936707),
 ('millennium', 0.6509759426116943),
 ('civilization', 0.6424446702003479),
 ('continent', 0.6343120336532593),
 ('world', 0.6335223913192749),
 ('mountain', 0.6271457076072693),
 ('moon', 0.626999020576477)]

In [15]:
glove.most_similar(positive=["balen", "country"])

[('rural', 0.5828554630279541),
 ('nepal', 0.561842143535614),
 ('internationally', 0.5401859283447266),
 ('unhcr', 0.5383888483047485),
 ('progress', 0.5374758243560791),
 ('zambia', 0.5372241139411926),
 ('lagging', 0.5251506567001343),
 ('statistics', 0.5237585306167603),
 ('unemployment', 0.5190934538841248),
 ('unrwa', 0.5175881385803223)]